# Ionic Dielectric Constant with CASTEP

The ionic or lattice response to an external electric field is one of the two contributions to the total "static" dielectric constant. The electronic contribution $\varepsilon_\infty$ is obtained from the optics calculation, but the ionic contribution $\varepsilon_\mathrm{ion}$ requires a calculation of its own. The total is their sum:

$$\varepsilon_\mathrm{static} = \varepsilon_\infty + \varepsilon_\mathrm{ion}$$

`solphin` provides methods to write input files and read output files to calculate this using CASTEP.

CASTEP reaches this quantity with an electric field linear-response calculation (`task : Efield`), which also runs the Γ-point phonon calculation the ionic contribution needs. Unlike VASP, there is no finite-field alternative, so two restrictions apply:

* **Linear response needs norm-conserving pseudopotentials.** `solphin`'s `dfpt` patch sets `species_pot : NCP19` for you.
* **Linear response is not available for hybrid functionals.** Use a GGA such as PBE, which is typically an acceptable approximation for the lattice response.

Pre-requisites:
* Geometry Optimisation/Structural Relaxation

## Calculation

1) Import the necessary libraries.

In [ ]:
import solphin.castep_inputs as castep_inputs
import solphin.dielectric as dielectric
import solphin.vasp_inputs as vasp_inputs

2) Define our computational regime. The cut-off and k-point spacing should match our converged values, but we use a GGA functional here rather than the hybrid used elsewhere in these tutorials.

In [ ]:
functional = "PBE"        # The DFT functional
cut_off_energy = 450      # Plane-wave cut-off in eV
kpoint_mp_spacing = 0.03  # Maximum k-point spacing in Angstrom^-1, no 2*pi factor

3) We need the structure to be at an equilibrium under this regime, so we run a new relaxation first. Starting from a previously relaxed structure, this should not take long.

* If you have been using a GGA functional already, you can skip this step.

* Here we reference the relaxed structure in the test data, and write to the untracked `workdir`. CASTEP cannot read a `CONTCAR`, so we load it with `vasp_inputs.read_structure_pmg`, which handles both formats.

In [ ]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

In [ ]:
castep_inputs.write_castep_calculation(
    structure=structure,
    recipe=functional,
    out_dir="workdir/ionic_dielectric_relax",
    patches=["relax_cell", "tight_relax"],
    kpoint_mp_spacing=kpoint_mp_spacing,
    user_param_settings={"cut_off_energy": cut_off_energy},
)

4) Run CASTEP in `workdir/ionic_dielectric_relax`.

* The relaxation matters more here than usual: the ionic dielectric constant comes from the phonon modes, and residual forces on the atoms produce imaginary modes and a meaningless tensor. That is why we use the `tight_relax` patch.

5) Load the relaxed structure and set up the electric field response calculation. A CASTEP geometry optimisation writes its final structure to `<seed>-out.cell`, which `read_structure_pmg` reads. `write_castep_calculation` returns the path of the `.cell` file it wrote, which carries the seedname you will need when running CASTEP.

In [ ]:
gga_structure = vasp_inputs.read_structure_pmg("workdir/ionic_dielectric_relax/Cu2GeS3-out.cell")

In [ ]:
efield_cell = castep_inputs.write_castep_calculation(
    structure=gga_structure,
    recipe=functional,
    out_dir="workdir/ionic_dielectric",
    patches=["dfpt"],
    kpoint_mp_spacing=kpoint_mp_spacing,
    user_param_settings={"cut_off_energy": cut_off_energy},
)

print(efield_cell)

6) Run the calculation the way your machine wants CASTEP run — for example:

```bash
   cd workdir/ionic_dielectric
   mpirun -np 16 castep.mpi Cu2GeS3
```

* No post-processing step is needed — unlike the optics workflow, this one does not go through OptaDOS.

7) Once the calculation is complete, we can parse the ionic dielectric constant.

CASTEP prints the optical ($f\to\infty$) and DC ($f=0$) permittivity tensors side by side in the `.castep` file; `solphin` takes the difference. Pass the directory and `solphin` will find the `.castep` file inside it, or pass the file directly. If the directory holds more than one, use `seedname` to pick:

In [ ]:
eps_ion, eps_ion_tensor = dielectric.parse_ionic_dielectric(
    "workdir/ionic_dielectric", code="castep", seedname="Cu2GeS3"
)

print(f"Ionic contribution: {eps_ion:.3f}")
print(eps_ion_tensor)

**IMPORTANT** You should take care to ensure the convergence of the dielectric constant with respect to the k-point sampling. This often requires a denser mesh than a regular calculation.